In [0]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import *

volume_path = "/Volumes/dataworkspace/default/data"

# Clean column headers to prevent Delta special-character errors
def clean_cols(df):
    for c in df.columns:
        clean_name = re.sub(r'[ ,;{}()\n\t=%/°]', '_', c)
        clean_name = re.sub(r'_+', '_', clean_name).strip('_')
        df = df.withColumnRenamed(c, clean_name)
    return df

# 1. Ingest Raw Datasets
raw_telemetry = clean_cols(
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{volume_path}/Iot telemetry.csv")
)

raw_events = clean_cols(
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{volume_path}/Event.csv")
)

raw_metadata = clean_cols(
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{volume_path}/Asset Metadata.csv")
)

print(f"Bronze Ingestion Complete: {raw_telemetry.count()} telemetry records, {raw_events.count()} events.")

Bronze Ingestion Complete: 8 telemetry records, 8 events.


In [0]:
# Parse Timestamps
df_staged = raw_telemetry.withColumn("parsed_timestamp", F.to_timestamp("timestamp"))

# Apply Quality Check Flags
df_flagged = (
    df_staged
    .withColumn("flag_null_mandatory", 
        F.when(
            F.col("parsed_timestamp").isNull() | 
            F.col("asset_id").isNull() | 
            (F.trim(F.col("asset_id")) == "") |
            F.col("site_id").isNull() | 
            F.col("building_id").isNull(), 1
        ).otherwise(0)
    )
    .withColumn("flag_schema_violation", 
        F.when(F.col("parsed_timestamp").isNull() & F.col("timestamp").isNotNull(), 1).otherwise(0)
    )
    .withColumn("flag_outlier", 
        F.when(
            (F.col("temperature_C") < -20) | (F.col("temperature_C") > 120) |
            (F.col("humidity") < 0) | (F.col("humidity") > 100) |
            (F.col("pressure_hPa") < 700) | (F.col("pressure_hPa") > 1300) |
            (F.col("vibration_mm_s") < 0) | (F.col("vibration_mm_s") > 50) |
            (F.col("power_consumption_kW") < 0), 1
        ).otherwise(0)
    )
)

# Deduplicate
duplicates_count = df_flagged.count() - df_flagged.dropDuplicates(["timestamp", "asset_id", "sensor_id"]).count()
df_deduped = df_flagged.dropDuplicates(["timestamp", "asset_id", "sensor_id"])

# Quality Decision Tagging
df_validated = (
    df_deduped
    .withColumn(
        "is_valid",
        F.when(
            (F.col("flag_null_mandatory") == 0) & 
            (F.col("flag_schema_violation") == 0) & 
            (F.col("flag_outlier") == 0), True
        ).otherwise(False)
    )
    .withColumn(
        "rejection_reason",
        F.concat_ws(", ",
            F.when(F.col("flag_null_mandatory") == 1, "NULL_MANDATORY_FIELD"),
            F.when(F.col("flag_schema_violation") == 1, "SCHEMA_TIMESTAMP_INVALID"),
            F.when(F.col("flag_outlier") == 1, "METRIC_OUTLIER_OUT_OF_BOUNDS")
        )
    )
)

# Split and Save Clean vs Quarantine
clean_telemetry = df_validated.filter(F.col("is_valid") == True).drop("is_valid", "rejection_reason")
quarantine_telemetry = df_validated.filter(F.col("is_valid") == False)

clean_telemetry.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dataworkspace.default.silver_telemetry_clean")
quarantine_telemetry.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dataworkspace.default.quarantine_telemetry")

# Generate Automated DQ Report
total_in = raw_telemetry.count()
total_clean = clean_telemetry.count()
total_quar = quarantine_telemetry.count()

dq_summary = [
    ("Total Records Ingested", total_in, "100.0%"),
    ("Duplicates Dropped", duplicates_count, f"{round((duplicates_count/max(total_in, 1))*100, 2)}%"),
    ("Passed Quality Gate (Clean)", total_clean, f"{round((total_clean/max(total_in, 1))*100, 2)}%"),
    ("Quarantined Records", total_quar, f"{round((total_quar/max(total_in, 1))*100, 2)}%")
]

# Replace lines 74-75 with:
df_dq_report = spark.createDataFrame(dq_summary, ["Metric", "Count", "Percentage"])

(
    df_dq_report.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dataworkspace.default.data_quality_summary_report")
)

In [0]:
# 1. Hourly Aggregations
hourly_curated = (
    clean_telemetry
    .withColumn("hour_window", F.date_trunc("hour", "parsed_timestamp"))
    .groupBy("site_id", "building_id", "asset_id", "hour_window")
    .agg(
        F.sum("power_consumption_kW").alias("hourly_energy_consumption_kWh"),
        F.avg("temperature_C").alias("avg_temperature_C"),
        F.avg("humidity").alias("avg_humidity_pct"),
        F.avg("pressure_hPa").alias("avg_pressure_hPa"),
        F.avg("vibration_mm_s").alias("avg_vibration")
    )
)

# 2. Fault Statistics
fault_statistics = (
    raw_events
    .withColumn("parsed_timestamp", F.to_timestamp("timestamp"))
    .filter(F.upper(F.col("event_type")).isin(["FAULT", "ERROR", "FAILURE"]))
    .groupBy("asset_id")
    .agg(
        F.count("event_id").alias("total_fault_count"),
        F.max("parsed_timestamp").alias("last_fault_timestamp")
    )
)

# 3. Gold Analytical Metrics
asset_metrics = (
    hourly_curated
    .groupBy("asset_id")
    .agg(
        F.sum("hourly_energy_consumption_kWh").alias("total_energy_kWh"),
        F.avg("avg_temperature_C").alias("overall_avg_temp_C"),
        F.avg("avg_humidity_pct").alias("overall_avg_humidity_pct")
    )
    .join(fault_statistics, on="asset_id", how="left")
    .fillna(0, subset=["total_fault_count"])
)

site_metrics = (
    hourly_curated
    .groupBy("site_id")
    .agg(
        F.sum("hourly_energy_consumption_kWh").alias("total_site_energy_kWh"),
        F.avg("avg_temperature_C").alias("site_avg_temp_C"),
        F.avg("avg_humidity_pct").alias("site_avg_humidity_pct"),
        F.countDistinct("building_id").alias("total_buildings"),
        F.countDistinct("asset_id").alias("total_assets")
    )
)

# Save Gold Tables
asset_metrics.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dataworkspace.default.asset_metrics")
site_metrics.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dataworkspace.default.site_metrics")

print("Gold Metrics Refreshed Successfully!")

Gold Metrics Refreshed Successfully!


In [0]:
%sql
-- Optimize Gold & Silver tables with Z-Ordering for high-speed dashboard reads
OPTIMIZE dataworkspace.default.silver_telemetry_clean ZORDER BY (asset_id, sensor_id);
OPTIMIZE dataworkspace.default.asset_metrics ZORDER BY (asset_id);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 2332), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1786805969361, 1786805969806, 8, 0, null, List(0, 0), null, 6, 6, 0, 0, null, null)"
